# Ablation Studies

Systematic analysis of model components and hyperparameters.

In [ ]:
import sys
sys.path.insert(0, '..')

from pytorch_lightning import seed_everything
from src.data import GestureDataModule
from src.ablation import AblationConfig, AblationRunner, BILSTM_ABLATIONS, TRANSFORMER_ABLATIONS
from src.ablation.config import BILSTM_QUICK, TRANSFORMER_QUICK

## Configuration

In [ ]:
MODEL = 'bilstm'  # 'bilstm' or 'transformer'
QUICK_MODE = True  # Use fewer configurations for faster results

DATA_PATH = '../data/DYLEM-GRID'
SEED = 42

seed_everything(SEED)

## Select Ablation Configuration

In [ ]:
if QUICK_MODE:
    config = BILSTM_QUICK if MODEL == 'bilstm' else TRANSFORMER_QUICK
else:
    config = BILSTM_ABLATIONS if MODEL == 'bilstm' else TRANSFORMER_ABLATIONS

print(f'Configuration: {config.name}')
print(f'Model: {config.model}')
print(f'Total configurations: {config.get_num_configurations()}')
print(f'CV: {config.n_folds}-fold' if config.use_cv else 'Single split')

print('\nAblation parameters:')
for param, values in config.ablations.items():
    print(f'  {param}: {values}')

## Run Ablation Study

In [ ]:
dm = GestureDataModule(data_path=DATA_PATH, seed=SEED)
runner = AblationRunner(config, dm, {'accelerator': 'auto'})

results = runner.run(verbose=True)

## Results Summary

In [ ]:
df = results.to_dataframe()
df

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

params = df['ablation_param'].unique()
n = len(params)
cols = min(2, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(7*cols, 5*rows))
if n == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for i, param in enumerate(params):
    ax = axes[i]
    p = df[df['ablation_param'] == param].copy()
    p = p.sort_values('ablation_value', key=lambda x: pd.to_numeric(x, errors='coerce'))
    
    x = range(len(p))
    bars = ax.bar(x, p['mean_accuracy'], yerr=p['std_accuracy'], capsize=5, color='steelblue')
    ax.set_xticks(x)
    ax.set_xticklabels(p['ablation_value'].astype(str), rotation=45, ha='right')
    ax.set_xlabel(param)
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Ablation: {param}')
    ax.set_ylim(0, 1.05)
    ax.axhline(p['mean_accuracy'].max(), color='red', linestyle='--', alpha=0.5)
    
    for bar, acc in zip(bars, p['mean_accuracy']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.3f}', ha='center', fontsize=9)

for i in range(n, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

## Save Results

In [ ]:
from pathlib import Path

save_dir = Path(f'../results/ablation/{config.model}')
results.save(str(save_dir))
config.to_yaml(str(save_dir / f'{config.name}_config.yaml'))

print(f'Results saved to {save_dir}')

## Best Configuration

In [ ]:
best = max(results.results, key=lambda r: r.mean_accuracy)

print('=' * 50)
print('BEST CONFIGURATION')
print('=' * 50)
print(f'Name: {best.config_name}')
print(f'Accuracy: {best.mean_accuracy:.4f} ± {best.std_accuracy:.4f}')
print(f'F1 Score: {best.mean_f1:.4f}')